<a href="https://colab.research.google.com/github/IdoiaUrra-TFG/TFG---SVMs-con-Kernels-Ortogonales-Un-Estudio-de-Sensibilidad-por-Grado/blob/main/TFG%2C_curva_A%2C_clases_desbalanceadas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#TFG: SVMs con Kernels Ortogonales - Estudio de Sensibilidad por Grado

##Caso A: Elipse (frontera de grado 2) con clases desbalanceadas

Este notebook implementa el experimento descrito en el capítulo 3 del TFG para el caso A con una diferencia en las clases, cuya frontera de decisión teórica es una elipse de grado 2.

In [ ]:
# Librerías necesarias para el experimento
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score
from sklearn import svm
from sklearn.preprocessing import MaxAbsScaler, MinMaxScaler
import scipy.stats as st
import matplotlib.pyplot as plt
import tabulate
from tabulate import tabulate

Se construye el kernel K_N^(2D) descrito en el Capítulo 2 del TFG usando polinomios de Chebyshev como familia de polinomios ortogonales.
Los pesos w_n se toman todos iguales a 1.

In [ ]:
# POLINOMIO DE CHEBYSHEV de grano n evaluado en x
# Calculado mediante la recurrencia: P_{n+1}(x) = 2xP_n(x)-P_{n-1}(x)
def pc( x, n):
  if n==0:
    return 1
  elif n==1:
    return x
  else:
    pc_anterior = x  # P_1
    pc_anteanterior = 1  # P_0
    for i in range(1,n):
      pc_actual = (2*x*pc_anterior - pc_anteanterior)  # P_{i+1}
      pc_anteanterior = pc_anterior
      pc_anterior = pc_actual
    return pc_actual

# KERNEL TRUNCADO 1D:  K_N^(1D)(x,z) = sum_{n=0}^{N} P_n(x)·P_n(z)
# Con grado máximo N=6 por defecto
def kc1d( x, z, n=6):
  solkc1d = 0
  for i in range(0,n+1):
    solkc1d += pc(x,i) * pc(z,i)
  return solkc1d

# KERNEL 2D POR PRODUCTO TENSORIAL: K_N^(2D)(X,Z) = K_N^(1D)(x1,z1) · K_N^(1D)(x2,z2)
def kc2d( X, Z, n=6):
  solkc2d = np.zeros((X.shape[0], Z.shape[0]))
  for i in range(X.shape[0]):
    for j in range(Z.shape[0]):
      solkc2d[i,j] = kc1d(X[i,0],Z[j,0],n)*kc1d(X[i,1], Z[j,1],n)
  return solkc2d

Para estudiar la contribución de cada grado d, se construyen versiones del kernel en las que se elimina el término de grado g en ambas coordenadas: K_N^(2D,-g)(X,Z) = K_N^(1D,-g)(x1,z1) · K_N^(1D,-g)(x2,z2) donde K_N^(1D,-g) es el kernel 1D sin el sumando correspondiente al grado g.

In [ ]:
# Kernel 1D sin grado g
def grupokc1d(g):
    def kc1d_sing( x, z, g=g, n=6):
      solkc1d = 0
      for i in [num for num in range(0, n+1) if num != g]:
        solkc1d += pc(x,i) * pc(z,i)
      return solkc1d
    return kc1d_sing

# Diccionario que almacena el kernel 1D sin grado g, para g en {1,...,6}
kc1dsingrado = {}

for g in range(1, 7):
    kc1dsingrado[g] = grupokc1d(g)

# Kernel 2D sin grado g
def grupokc2d(g):
  def kc2d_sing( X, Z, g=g, n=6):
    solkc2d = np.zeros((X.shape[0], Z.shape[0]))
    for i in range(X.shape[0]):
      for j in range(Z.shape[0]):
        solkc2d[i,j] = kc1dsingrado[g](X[i,0],Z[j,0])*kc1dsingrado[g](X[i,1], Z[j,1])
    return solkc2d
  return kc2d_sing

# Diccionario que almacena el kernel 2D sin grado g, para g en {1,...,6}
kc2dsingrado = {}

for g in range(1,7):
  kc2dsingrado[g] = grupokc2d(g)



La frontera teórica es una elipse de parámetros a=0.9, b=0.6. Se generan 250 puntos: 75 para la clase +1 y 175 para la clase -1. El dataset se genera una sola vez y se escala globalmente a [-1,1].

In [ ]:
# ELIPSE con a = 0.9 y b = 0.6
# Regla de etiquetado: clase +1 si el punto está dentro de la elipse, -1 si está fuera
def curvaA(x,y):
    elipse = (x**2)/(0.9**2) + (y**2)/(0.6**2) -1
    if elipse > 0:
      clase = -1
    else:
      clase = 1
    return clase


#Generamos datos de la clase +1
datos1 = []
while len(datos1) < 75:
  x = np.random.uniform(-1.2, 1.2) + np.random.normal( loc= 0.0, scale = 0.03)
  y = np.random.uniform(-1.2, 1.2) + np.random.normal( loc= 0.0, scale = 0.03)
  if curvaA(x,y) == 1:
    datos1.append([x,y])

#Generamos datos de la clase -1
datos2 = []
while len(datos2) < 175:
    x = np.random.uniform(-1.2, 1.2) + np.random.normal( loc= 0.0, scale = 0.03)
    y = np.random.uniform(-1.2, 1.2) + np.random.normal( loc= 0.0, scale = 0.03)
    if curvaA(x,y) == -1:
      datos2.append([x,y])


datos = np.array(np.vstack([datos1,datos2]))
clases = np.array([+1]*75 + [-1]*175)

# Escalado MinMax a [-1, 1] ajustado sobre el dataset completo
escalar = MinMaxScaler(feature_range = (-1,1))
escalar.fit(datos)
datos_escalado = escalar.transform(datos)


In [ ]:
# Bucle principal

# Se crean diccionarios indexados por grado (1 a 6) para acumular,
# a lo largo de las 10 repeticiones, las caídas de rendimiento y las
# diferencias en número de vectores soporte al apagar cada grado.


# Caída absoluta de F1 y accuracy al apagar el grado g: Delta(g;r) = metrica_completo - metrica_apagado
caidarendimientof1 = {}
for g in range(1,7):
  caidarendimientof1[ g] = []

caidarendimientoacc = {}
for g in range(1,7):
  caidarendimientoacc[ g] = []

# Modelos entrenados en la repetición r=1, guardados para visualizar fronteras de decisión
clf_F1_apagado_r1 = {}
for g in range(1,7):
  clf_F1_apagado_r1[ g] = []

clf_ACC_apagado_r1 = {}
for g in range(1,7):
  clf_ACC_apagado_r1[ g] = []

# Diferencia en número de vectores soporte entre el modelo completo y el apagado
SVM_F1 = {}
for g in range(1,7):
  SVM_F1[ g] = []

SVM_ACC = {}
for g in range(1,7):
  SVM_ACC[ g] = []

# F1 y accuracy del modelo completo en cada repetición
f1total = []
acctotal = []

# C* seleccionado en cada repetición (uno por métrica, por si difieren)
mejores_C_f1 = []
mejores_C_acc = []


# Bucle: R=10 repeticiones
for seed in range(1,11):

  # Partición estratificada train/test (70%/30%)
  Datos_train_Escalado, Datos_test_Escalado, Clases_train, Clases_test = train_test_split(datos_escalado, clases, test_size=0.3, random_state = seed )

  # Selección del hiperparámetro C* por validación cruzada 5-fold sobre train
  # Se evalúan 5 valores de C sobre la rejilla fijada en el protocolo.
  # Se seleccionan por separado el mejor C para F1 y para accuracy,
  # para poder estudiar ambas métricas de forma independiente.
  prm_C = [0.01, 0.1, 1, 10, 100]

  X_train = Datos_train_Escalado
  y_train = Clases_train
  X_test  = Datos_test_Escalado
  y_test  = Clases_test

  cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

  mejor_C_f1 = None
  mejor_C_acc = None
  mejor_f1 = -1
  mejor_acc = -1

  for C in prm_C:
      f1_scores = []
      acc_scores = []

      for indice_train, indice_val in cv.split(X_train, y_train):
          Datos_tr, Datos_val = X_train[indice_train], X_train[indice_val]
          Clases_tr, Clases_val = y_train[indice_train], y_train[indice_val]

          clf = svm.SVC(kernel=kc2d, C=C)   # Modelo completo con el kernel 2D
          clf.fit(Datos_tr, Clases_tr)
          Clases_pred = clf.predict(Datos_val)

          # Evaluación del modelo completo
          f1_scores.append(f1_score(Clases_val, Clases_pred))
          acc_scores.append(accuracy_score(Clases_val, Clases_pred))

      f1_mean = np.mean(f1_scores)
      acc_mean = np.mean(acc_scores)


      if f1_mean > mejor_f1:
          mejor_f1 = f1_mean
          mejor_C_f1 = C

      if acc_mean > mejor_acc:
          mejor_acc = acc_mean
          mejor_C_acc = C

  mejores_C_f1.append(mejor_C_f1)
  mejores_C_acc.append(mejor_C_acc)

  # Se distinguen dos casos según si el C* óptimo para F1 y para accuracy coincide o no.
  if mejor_C_f1 == mejor_C_acc:
    print(f"Seed {seed}: La C con mayor rendimiento para las métricas F1 y accuracy es la misma y es C={mejor_C_f1}")

    # Modelo completo (único C*, válido para ambas métricas)
    clf_completo = svm.SVC(kernel = kc2d, C = mejor_C_f1)
    clf_completo.fit(Datos_train_Escalado, Clases_train)
    y_pred = clf_completo.predict(Datos_test_Escalado)

    f1 = f1_score(Clases_test, y_pred)
    f1total.append(f1)
    acc= accuracy_score(Clases_test, y_pred)
    acctotal.append(acc)

    # Modelos con apagado de cada grado g ∈ {1,...,6}
    # Se usa el mismo C* y los mismos datos que el modelo completo
    for grado in range(1,7):
      clf_apagado = svm.SVC(kernel = kc2dsingrado[grado], C = mejor_C_f1)
      clf_apagado.fit(Datos_train_Escalado, Clases_train)
      y_pred_singrado = clf_apagado.predict( Datos_test_Escalado)

      f1_singrado = f1_score(Clases_test, y_pred_singrado)
      acc_singrado = accuracy_score(Clases_test, y_pred_singrado)

      # Se guarda: (semilla, Delta absoluta, Delta relativa)
      caidarendimientof1[grado].append((seed, f1 - f1_singrado, (f1 - f1_singrado)/f1))
      caidarendimientoacc[grado].append((seed, acc - acc_singrado, (acc - acc_singrado)/acc))

      # Diferencia en número de vectores soporte entre modelo completo y apagado
      SVM_F1[grado].append((seed, int(sum(clf_completo.n_support_)) - int(sum(clf_apagado.n_support_))))
      SVM_ACC[grado].append((seed, int(sum(clf_completo.n_support_)) - int(sum(clf_apagado.n_support_))))

      # Se guardan los modelos de r=1 para visualizar las fronteras de decisión
      if seed == 1:
        Datos_test_r1 = Datos_test_Escalado
        Clases_test_r1 = Clases_test
        clf_F1_completo_r1 = clf_completo
        clf_ACC_completo_r1 = clf_completo
        for gradosguardados in range(1,7):
          if grado == gradosguardados:
            clf_F1_apagado_r1[gradosguardados] = clf_apagado
            clf_ACC_apagado_r1[gradosguardados] = clf_apagado

  else:
    print(f"Seed {seed}: La C con mayor rendimiento para la métrica F1 es C={mejor_C_f1} y para la métrica accuracy es C={mejor_C_acc}")

    # Dos modelos completos: uno optimizado para F1 y otro para accuracy
    clf_F1_completo = svm.SVC(kernel = kc2d, C = mejor_C_f1)
    clf_F1_completo.fit(Datos_train_Escalado, Clases_train)
    y_pred_F1 = clf_F1_completo.predict(Datos_test_Escalado)

    clf_ACC_completo = svm.SVC(kernel = kc2d, C = mejor_C_acc)
    clf_ACC_completo.fit(Datos_train_Escalado, Clases_train)
    y_pred_ACC = clf_ACC_completo.predict(Datos_test_Escalado)

    f1 = f1_score(Clases_test, y_pred_F1)
    f1total.append(f1)
    acc = accuracy_score(Clases_test, y_pred_ACC)
    acctotal.append(acc)

    # Modelos apagados: uno por métrica, cada uno con su C* correspondiente
    # Para cada métrica se procede de la misma manera que en el modelo completo
    for grado in range(1,7):
      clf_F1_apagado = svm.SVC(kernel = kc2dsingrado[grado], C = mejor_C_f1)
      clf_F1_apagado.fit(Datos_train_Escalado, Clases_train)
      y_pred_singrado_F1 = clf_F1_apagado.predict( Datos_test_Escalado)

      clf_ACC_apagado = svm.SVC(kernel = kc2dsingrado[grado], C = mejor_C_acc)
      clf_ACC_apagado.fit(Datos_train_Escalado, Clases_train)
      y_pred_singrado_ACC = clf_ACC_apagado.predict( Datos_test_Escalado)

      f1_singrado = f1_score(Clases_test, y_pred_singrado_F1)
      acc_singrado = accuracy_score(Clases_test, y_pred_singrado_ACC)

      caidarendimientof1[grado].append((seed, f1 - f1_singrado, (f1 - f1_singrado)/f1))
      caidarendimientoacc[grado].append((seed, acc - acc_singrado, (acc - acc_singrado)/acc))

      SVM_F1[grado].append((seed, int(sum(clf_F1_completo.n_support_)) - int(sum(clf_F1_apagado.n_support_))))
      SVM_ACC[grado].append((seed, int(sum(clf_ACC_completo.n_support_)) - int(sum(clf_ACC_apagado.n_support_))))

      if seed == 1:
        Datos_test_r1 = Datos_test_Escalado
        Clases_test_r1 = Clases_test
        clf_F1_completo_r1 = clf_F1_completo
        clf_ACC_completo_r1 = clf_ACC_completo
        for gradosguardados in range(1,7):
          if grado == gradosguardados:
            clf_F1_apagado_r1[gradosguardados] = clf_F1_apagado
            clf_ACC_apagado_r1[gradosguardados] = clf_ACC_apagado


In [ ]:
# Resumen del C* seleccionado en cada repetición
output_lines = []
for i, seed in enumerate(range(1, 11)):
    if mejores_C_f1[i] == mejores_C_acc[i]:
        output_lines.append(f"Seed {seed}: La C con mayor rendimiento para las métricas F1 y accuracy es la misma y es C={mejores_C_f1[i]}")
    else:
        output_lines.append(f"Seed {seed}: La C con mayor rendimiento para la métrica F1 es C={mejores_C_f1[i]} y para la métrica accuracy es C={mejores_C_acc[i]}")



texto = "\n".join(output_lines)

# Se guarda como imagen para facilitar su inclusión en la memoria
fig, ax = plt.subplots(figsize=(6, len(output_lines) * 0.4 + 1))
ax.axis('off')
ax.text(0.01, 0.99, texto, transform=ax.transAxes,
        fontsize=11, verticalalignment='top', fontfamily='monospace')

plt.tight_layout()
plt.savefig("resultados_prints.png", dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Media, desviación típica e intervalos de confianza
mediaf1 = {}
desviacionf1 = {}
intervalof1 = {}

mediaacc = {}
desviacionacc = {}
intervaloacc = {}

for r in range(1,7):
  mediaf1[r] = []
  desviacionf1[r] = []
  intervalof1[r] = []

  mediaacc[r] = []
  desviacionacc[r] = []
  intervaloacc[r] = []

# Se extraen las 10 caídas absolutas Delta(g;r) de F1 y accuracy
for g in range(1,7):
  totvariacionesf1 = []
  totvariacionesacc = []
  for r in range(10):
    totvariacionesf1.append(caidarendimientof1[g][r][1])
    totvariacionesacc.append(caidarendimientoacc[g][r][1])

  # Media de las 10 caídas
  mediaf1[g] = float(np.mean(totvariacionesf1))
  mediaacc[g] = float(np.mean(totvariacionesacc))

  # Desviación estándar
  desviacionf1[g] = float(np.std(totvariacionesf1))
  desviacionacc[g] = float(np.std(totvariacionesacc))

  # Intervalo de confianza al 95%
  # Con la distribución t de Student con R-1=9 grados de libertad
  a, b = st.t.interval (confidence = 0.95, df = len(totvariacionesf1)-1, loc = mediaf1[g], scale = st.sem (totvariacionesf1))
  intervalof1[g] = (float(a), float(b))
  a, b = st.t.interval (confidence = 0.95, df = len(totvariacionesacc)-1, loc = mediaacc[g], scale = st.sem (totvariacionesacc))
  intervaloacc[g] = (float(a), float(b))

In [ ]:
# Grado crítico: d* es aquel cuyo apagado produce, en promedio, la mayor caída de rendimiento
mejor_mediaf1 = mediaf1[1]
for g in range(1,7):
    if mediaf1[g] >= mejor_mediaf1:
        mejor_mediaf1 = mediaf1[g]
        grado_criticof1 = g

mejor_mediaacc = mediaacc[1]
for g in range(1,7):
    if mediaacc[g] >= mejor_mediaacc:
        mejor_mediaacc = mediaacc[g]
        grado_criticoacc = g
print(grado_criticof1, grado_criticoacc)

In [ ]:
# Tabla resumen: media, desviación típica e IC95% de F1 y de Accuracy para cada grado apagado
datostabla = []
for g in range(1,7):
  datostabla.append([g, mediaf1[g], desviacionf1[g], intervalof1[g]])

table = tabulate(
    datostabla,
    headers=["Grado", "Media de la diferencia de F1", "Desviación estándar de la diferencia de F1", "Intervalo de Confianza de la diferencia de F1"],
    tablefmt="fancy_grid")

datostabla2 = []
for g in range(1,7):
  datostabla2.append([g, mediaacc[g], desviacionacc[g], intervaloacc[g]])

table2 = tabulate(
    datostabla2,
    headers=["Grado", "Media de la diferencia de Acc", "Desviación estándar de la diferencia de Acc", "Intervalo de Confianza de la diferencia de Acc"],
    tablefmt="fancy_grid")

print(table)
print(table2)


In [ ]:
# Se guarda la tabla como imagen para facilitar su inclusión en la memoria
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(18, len(lines) * 0.35))
ax.axis('off')
ax.text(0.01, 0.99, table + "\n\n" + table2, transform=ax.transAxes,
        fontsize=8, verticalalignment='top', fontfamily='monospace')

plt.tight_layout()
plt.savefig("tablas.png", dpi=150, bbox_inches='tight')
plt.show()

from google.colab import files
files.download("tablas.png")

In [ ]:
# Análisis de cómo varía el número de Vectores Soporte al apagar cada grado
media_svmf1 = {}
desviacion_svmf1 = {}
intervalo_svmf1 = {}

media_svmacc = {}
desviacion_svmacc = {}
intervalo_svmacc = {}

for r in range(1,7):
  media_svmf1[r] = []
  desviacion_svmf1[r] = []
  intervalo_svmf1[r] = []

  media_svmacc[r] = []
  desviacion_svmacc[r] = []
  intervalo_svmacc[r] = []

for g in range(1,7):
  totvariaciones_svmf1 = []
  totvariaciones_svmacc = []
  for r in range(10):
    totvariaciones_svmf1.append(SVM_F1[g][r][1])
    totvariaciones_svmacc.append(SVM_ACC[g][r][1])

  # Media
  media_svmf1[g] = float(np.mean(totvariaciones_svmf1))
  media_svmacc[g] = float(np.mean(totvariaciones_svmacc))

  # Desviación estándar
  desviacion_svmf1[g] = float(np.std(totvariaciones_svmf1))
  desviacion_svmacc[g] = float(np.std(totvariaciones_svmacc))

  # Intervalo de confianza
  a, b = st.t.interval (confidence = 0.95, df = len(totvariaciones_svmf1)-1, loc = media_svmf1[g], scale = st.sem (totvariaciones_svmf1))
  intervalo_svmf1[g] = (float(a), float(b)) #Para que me salgan como números y no como np.float64(-0.92....)
  a, b = st.t.interval (confidence = 0.95, df = len(totvariaciones_svmacc)-1, loc = media_svmacc[g], scale = st.sem (totvariaciones_svmacc))
  intervalo_svmacc[g] = (float(a), float(b)) #Para que me salgan como números y no como np.float64(-0.92....)

In [ ]:
# Tabla resumen de la diferencia media en número de vectores soporte
# entre el modelo completo y cada modelo con un grado apagado
datostabla = []
for g in range(1,7):
  datostabla.append([g, media_svmf1[g], desviacion_svmf1[g], intervalo_svmf1[g]])

table3 = tabulate(
    datostabla,
    headers=["Grado", "Media de la diferencia de SVM F1", "Desviación estándar de la diferencia de SVM F1", "Intervalo de Confianza de la diferencia de SVM F1"],
    tablefmt="fancy_grid")

datostabla2 = []
for g in range(1,7):
  datostabla2.append([g, media_svmacc[g], desviacion_svmacc[g], intervalo_svmacc[g]])

table4 = tabulate(
    datostabla2,
    headers=["Grado", "Media de la diferencia de SVM Acc", "Desviación estándar de la diferencia de SVM Acc", "Intervalo de Confianza de la diferencia de SVM Acc"],
    tablefmt="fancy_grid")

print(table3)
print(table4)

In [ ]:
# Se guarda la tabla como imagen para facilitar su inclusión en la memoria
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(18, len(lines) * 0.35))
ax.axis('off')
ax.text(0.01, 0.99, table3 + "\n\n" + table4, transform=ax.transAxes,
        fontsize=8, verticalalignment='top', fontfamily='monospace')

plt.tight_layout()
plt.savefig("tablassvm.png", dpi=150, bbox_inches='tight')
plt.show()

from google.colab import files
files.download("tablassvm.png")

In [ ]:
# GRÁFICO COMPARATIVO F1 vs ACC
# Se guarda la tabla como imagen para facilitar su inclusión en la memoria
grados = np.array([1, 2, 3, 4, 5, 6])

ic_inff1 = np.array([intervalof1[1][0], intervalof1[2][0], intervalof1[3][0], intervalof1[4][0], intervalof1[5][0], intervalof1[6][0]])
ic_supf1 = np.array([intervalof1[1][1], intervalof1[2][1], intervalof1[3][1], intervalof1[4][1], intervalof1[5][1], intervalof1[6][1]])
mediasf1 = np.array([mediaf1[1], mediaf1[2], mediaf1[3], mediaf1[4], mediaf1[5], mediaf1[6]])
alturasf1 = ic_supf1 - ic_inff1

ic_infacc = np.array([intervaloacc[1][0], intervaloacc[2][0], intervaloacc[3][0], intervaloacc[4][0], intervaloacc[5][0], intervaloacc[6][0]])
ic_supacc = np.array([intervaloacc[1][1], intervaloacc[2][1], intervaloacc[3][1], intervaloacc[4][1], intervaloacc[5][1], intervaloacc[6][1]])
mediasacc = np.array([mediaacc[1], mediaacc[2], mediaacc[3], mediaacc[4], mediaacc[5], mediaacc[6]])
alturasacc = ic_supacc - ic_infacc

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# F1
axes[0].bar(grados, alturasf1, bottom=ic_inff1, color="skyblue", edgecolor="black", alpha=0.8)
for x, m, inf, sup in zip(grados, mediasf1, ic_inff1, ic_supf1):
    axes[0].hlines(m, x-0.4, x+0.4, colors="red", linewidth=2)
    axes[0].text(x, m + 0.003, f"{m:.3f}", ha="center", va="bottom", fontsize=7, color="red")
    axes[0].text(x, inf + 0.002, f"{inf:.3f}", ha="center", va="bottom", fontsize=7, color="darkblue")
    axes[0].text(x, sup + 0.001, f"{sup:.3f}", ha="center", va="bottom", fontsize=7, color="darkblue")
axes[0].set_title("Intervalos F1")
axes[0].set_xlabel("Grado d")
axes[0].set_ylabel("Valor")
axes[0].grid(axis="y", alpha=0.3)

# ACC
axes[1].bar(grados, alturasacc, bottom=ic_infacc, color="lightgreen", edgecolor="black", alpha=0.8)
for x, m, inf, sup in zip(grados, mediasacc, ic_infacc, ic_supacc):
    axes[1].hlines(m, x-0.4, x+0.4, colors="red", linewidth=2)
    axes[1].text(x, m + 0.003, f"{m:.3f}", ha="center", va="bottom", fontsize=7, color="red")
    axes[1].text(x, inf + 0.002, f"{inf:.3f}", ha="center", va="bottom", fontsize=7, color="darkgreen")
    axes[1].text(x, sup + 0.001, f"{sup:.3f}", ha="center", va="bottom", fontsize=7, color="darkgreen")
axes[1].set_title("Intervalos ACC")
axes[1].set_xlabel("Grado d")
axes[1].grid(axis="y", alpha=0.3)

plt.suptitle("Comparación de intervalos F1 vs ACC")
plt.tight_layout()
plt.savefig("grafico_f1_acc.png", dpi=150, bbox_inches='tight')

from google.colab import files
files.download("grafico_f1_acc.png")
plt.show()

In [ ]:
# Visualización de Fronteras de decisión
# La región azul corresponde a la clase -1 y la roja a la clase +1
def plot_frontera(clf, X, y, nombre_archivo, titulo=""):
    # Se genera una malla densa sobre el espacio de entrada para evaluar el clasificador
    xx, yy = np.meshgrid(np.linspace(-1.1, 1.1, 350), np.linspace(-1.1, 1.1, 350))
    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = clf.predict(grid).reshape(xx.shape)

    plt.figure(figsize=(6, 5))
    plt.contourf(xx, yy, Z, alpha=0.3, cmap='bwr')
    plt.contour(xx, yy, Z, levels=[0], colors='k', linewidths=2)
    plt.scatter(X[:,0], X[:,1], c=y, cmap='bwr', edgecolors='k', s=30)
    if titulo:
        plt.title(titulo)
    plt.savefig(f"{nombre_archivo}.png", dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Frontera de decisión modelo completo de la repetición r=1
from google.colab import files

if mejores_C_f1[0] == mejores_C_acc[0]:
    plot_frontera(clf_F1_completo_r1, Datos_test_r1, Clases_test_r1, "Modelo completo (r=1) Y F1=ACC")
    files.download("Modelo completo (r=1) Y F1=ACC.png")
else:
    plot_frontera(clf_F1_completo_r1, Datos_test_r1, Clases_test_r1, "Modelo completo (r=1) F1")
    files.download("Modelo completo (r=1) F1.png")

    plot_frontera(clf_ACC_completo_r1, Datos_test_r1, Clases_test_r1, "Modelo completo (r=1) ACC")
    files.download("Modelo completo (r=1) ACC.png")

In [ ]:
# Frontera de decisión modelo con grado crítico d* apagado de la repetición r=1
from google.colab import files

if grado_criticof1 == grado_criticoacc:
    if mejores_C_f1[0] == mejores_C_acc[0]:
        plot_frontera(clf_F1_apagado_r1[grado_criticof1], Datos_test_r1, Clases_test_r1, f"Apagado d*={grado_criticof1} r=1 y F1 = ACC")
        files.download(f"Apagado d*={grado_criticof1} r=1 y F1 = ACC.png")
    else:
        plot_frontera(clf_F1_apagado_r1[grado_criticof1], Datos_test_r1, Clases_test_r1, f"Apagado d*={grado_criticof1} r=1 F1")
        files.download(f"Apagado d*={grado_criticof1} r=1 F1.png")
        plot_frontera(clf_ACC_apagado_r1[grado_criticoacc], Datos_test_r1, Clases_test_r1, f"Apagado d*={grado_criticoacc} r=1 ACC")
        files.download(f"Apagado d*={grado_criticoacc} r=1 ACC.png")
else:
    if mejores_C_f1[0] == mejores_C_acc[0]:
        plot_frontera(clf_F1_apagado_r1[grado_criticof1], Datos_test_r1, Clases_test_r1, f"Apagado d*={grado_criticof1} r=1 y F1 = ACC")
        files.download(f"Apagado d*={grado_criticof1} r=1 y F1 = ACC.png")
        plot_frontera(clf_F1_apagado_r1[grado_criticoacc], Datos_test_r1, Clases_test_r1, f"Apagado d*={grado_criticoacc} r=1 y F1 = ACC")
        files.download(f"Apagado d*={grado_criticoacc} r=1 y F1 = ACC.png")
    else:
        plot_frontera(clf_F1_apagado_r1[grado_criticof1], Datos_test_r1, Clases_test_r1, f"Apagado d*={grado_criticof1} r=1 F1")
        files.download(f"Apagado d*={grado_criticof1} r=1 F1.png")
        plot_frontera(clf_ACC_apagado_r1[grado_criticoacc], Datos_test_r1, Clases_test_r1, f"Apagado d*={grado_criticoacc} r=1 ACC")
        files.download(f"Apagado d*={grado_criticoacc} r=1 ACC.png")

In [ ]:
#Frontera de decisión modelo con grado 6 apagado de la repetición r=1
from google.colab import files

if mejores_C_f1[0] == mejores_C_acc[0]:
    plot_frontera(clf_F1_apagado_r1[6], Datos_test_r1, Clases_test_r1, "Apagado d=6 r=1 y F1 = ACC")
    files.download("Apagado d=6 r=1 y F1 = ACC.png")
else:
    plot_frontera(clf_F1_apagado_r1[6], Datos_test_r1, Clases_test_r1, "Apagado d=6 r=1 F1")
    files.download("Apagado d=6 r=1 F1.png")
    plot_frontera(clf_ACC_apagado_r1[6], Datos_test_r1, Clases_test_r1, "Apagado d=6 r=1 ACC")
    files.download("Apagado d=6 r=1 ACC.png")

In [ ]:
# Número de SVMs utilizados en los modelos
if mejores_C_f1[0] == mejores_C_acc[0]:
    print(f"Modelo completo - Nº Vectores soporte: {sum(clf_F1_completo_r1.n_support_)} ({clf_F1_completo_r1.n_support_[0]} pertenecientes a la clase -1, {clf_F1_completo_r1.n_support_[1]} pertenecientes a la clase +1).")
    if grado_criticof1 == grado_criticoacc:
        print(f"Grado d={grado_criticof1} apagado - Nº Vectores soporte: {sum(clf_F1_apagado_r1[grado_criticof1].n_support_)} ({clf_F1_apagado_r1[grado_criticof1].n_support_[0]} pertenecientes a la clase -1, {clf_F1_apagado_r1[grado_criticof1].n_support_[1]} pertenecientes a la clase +1).")
    else:
        print(f"Grado d={grado_criticof1} apagado (F1) - Nº Vectores soporte: {sum(clf_F1_apagado_r1[grado_criticof1].n_support_)} ({clf_F1_apagado_r1[grado_criticof1].n_support_[0]} pertenecientes a la clase -1, {clf_F1_apagado_r1[grado_criticof1].n_support_[1]} pertenecientes a la clase +1).")
        print(f"Grado d={grado_criticoacc} apagado (ACC) - Nº Vectores soporte: {sum(clf_F1_apagado_r1[grado_criticoacc].n_support_)} ({clf_F1_apagado_r1[grado_criticoacc].n_support_[0]} pertenecientes a la clase -1, {clf_F1_apagado_r1[grado_criticoacc].n_support_[1]} pertenecientes a la clase +1).")
    print(f"Grado d=6 apagado - Nº Vectores soporte: {sum(clf_F1_apagado_r1[6].n_support_)} ({clf_F1_apagado_r1[6].n_support_[0]} pertenecientes a la clase -1, {clf_F1_apagado_r1[6].n_support_[1]} pertenecientes a la clase +1).")
else:
    print(f"Modelo completo (F1) - Nº Vectores soporte: {sum(clf_F1_completo_r1.n_support_)} ({clf_F1_completo_r1.n_support_[0]} pertenecientes a la clase -1, {clf_F1_completo_r1.n_support_[1]} pertenecientes a la clase +1).")
    print(f"Modelo completo (ACC) - Nº Vectores soporte: {sum(clf_ACC_completo_r1.n_support_)} ({clf_ACC_completo_r1.n_support_[0]} pertenecientes a la clase -1, {clf_ACC_completo_r1.n_support_[1]} pertenecientes a la clase +1).")
    if grado_criticof1 == grado_criticoacc:
        print(f"Grado d={grado_criticof1} apagado (F1) - Nº Vectores soporte: {sum(clf_F1_apagado_r1[grado_criticof1].n_support_)} ({clf_F1_apagado_r1[grado_criticof1].n_support_[0]} pertenecientes a la clase -1, {clf_F1_apagado_r1[grado_criticof1].n_support_[1]} pertenecientes a la clase +1).")
        print(f"Grado d={grado_criticoacc} apagado (ACC) - Nº Vectores soporte: {sum(clf_ACC_apagado_r1[grado_criticoacc].n_support_)} ({clf_ACC_apagado_r1[grado_criticoacc].n_support_[0]} pertenecientes a la clase -1, {clf_ACC_apagado_r1[grado_criticoacc].n_support_[1]} pertenecientes a la clase +1).")
    else:
        print(f"Grado d={grado_criticof1} apagado (F1) - Nº Vectores soporte: {sum(clf_F1_apagado_r1[grado_criticof1].n_support_)} ({clf_F1_apagado_r1[grado_criticof1].n_support_[0]} pertenecientes a la clase -1, {clf_F1_apagado_r1[grado_criticof1].n_support_[1]} pertenecientes a la clase +1).")
        print(f"Grado d={grado_criticoacc} apagado (ACC) - Nº Vectores soporte: {sum(clf_ACC_apagado_r1[grado_criticoacc].n_support_)} ({clf_ACC_apagado_r1[grado_criticoacc].n_support_[0]} pertenecientes a la clase -1, {clf_ACC_apagado_r1[grado_criticoacc].n_support_[1]} pertenecientes a la clase +1).")
    print(f"Grado d=6 apagado (F1) - Nº Vectores soporte: {sum(clf_F1_apagado_r1[6].n_support_)} ({clf_F1_apagado_r1[6].n_support_[0]} pertenecientes a la clase -1, {clf_F1_apagado_r1[6].n_support_[1]} pertenecientes a la clase +1).")
    print(f"Grado d=6 apagado (ACC) - Nº Vectores soporte: {sum(clf_ACC_apagado_r1[6].n_support_)} ({clf_ACC_apagado_r1[6].n_support_[0]} pertenecientes a la clase -1, {clf_ACC_apagado_r1[6].n_support_[1]} pertenecientes a la clase +1).")

In [ ]:
# Se guarda la imagen para facilitar su inclusión en la memoria
import matplotlib.pyplot as plt

lines = []
if mejores_C_f1[0] == mejores_C_acc[0]:
    lines.append(f"Modelo completo      - Nº Vectores soporte: {sum(clf_F1_completo_r1.n_support_)} ({clf_F1_completo_r1.n_support_[0]} pertenecientes a la clase -1, {clf_F1_completo_r1.n_support_[1]} pertenecientes a la clase +1).")
    if grado_criticof1 == grado_criticoacc:
        lines.append(f"Grado d={grado_criticof1} apagado - Nº Vectores soporte: {sum(clf_F1_apagado_r1[grado_criticof1].n_support_)} ({clf_F1_apagado_r1[grado_criticof1].n_support_[0]} pertenecientes a la clase -1, {clf_F1_apagado_r1[grado_criticof1].n_support_[1]} pertenecientes a la clase +1).")
    else:
        lines.append(f"Grado d={grado_criticof1} apagado (F1) - Nº Vectores soporte: {sum(clf_F1_apagado_r1[grado_criticof1].n_support_)} ({clf_F1_apagado_r1[grado_criticof1].n_support_[0]} pertenecientes a la clase -1, {clf_F1_apagado_r1[grado_criticof1].n_support_[1]} pertenecientes a la clase +1).")
        lines.append(f"Grado d={grado_criticoacc} apagado (ACC) - Nº Vectores soporte: {sum(clf_F1_apagado_r1[grado_criticoacc].n_support_)} ({clf_F1_apagado_r1[grado_criticoacc].n_support_[0]} pertenecientes a la clase -1, {clf_F1_apagado_r1[grado_criticoacc].n_support_[1]} pertenecientes a la clase +1).")
    lines.append(f"Grado d=6 apagado - Nº Vectores soporte: {sum(clf_F1_apagado_r1[6].n_support_)} ({clf_F1_apagado_r1[6].n_support_[0]} pertenecientes a la clase -1, {clf_F1_apagado_r1[6].n_support_[1]} pertenecientes a la clase +1).")
else:
    lines.append(f"Modelo completo (F1) - Nº Vectores soporte: {sum(clf_F1_completo_r1.n_support_)} ({clf_F1_completo_r1.n_support_[0]} pertenecientes a la clase -1, {clf_F1_completo_r1.n_support_[1]} pertenecientes a la clase +1).")
    lines.append(f"Modelo completo (ACC) - Nº Vectores soporte: {sum(clf_ACC_completo_r1.n_support_)} ({clf_ACC_completo_r1.n_support_[0]} pertenecientes a la clase -1, {clf_ACC_completo_r1.n_support_[1]} pertenecientes a la clase +1).")
    if grado_criticof1 == grado_criticoacc:
        lines.append(f"Grado d={grado_criticof1} apagado (F1) - Nº Vectores soporte: {sum(clf_F1_apagado_r1[grado_criticof1].n_support_)} ({clf_F1_apagado_r1[grado_criticof1].n_support_[0]} pertenecientes a la clase -1, {clf_F1_apagado_r1[grado_criticof1].n_support_[1]} pertenecientes a la clase +1).")
        lines.append(f"Grado d={grado_criticoacc} apagado (ACC) - Nº Vectores soporte: {sum(clf_ACC_apagado_r1[grado_criticoacc].n_support_)} ({clf_ACC_apagado_r1[grado_criticoacc].n_support_[0]} pertenecientes a la clase -1, {clf_ACC_apagado_r1[grado_criticoacc].n_support_[1]} pertenecientes a la clase +1).")
    else:
        lines.append(f"Grado d={grado_criticof1} apagado (F1) - Nº Vectores soporte: {sum(clf_F1_apagado_r1[grado_criticof1].n_support_)} ({clf_F1_apagado_r1[grado_criticof1].n_support_[0]} pertenecientes a la clase -1, {clf_F1_apagado_r1[grado_criticof1].n_support_[1]} pertenecientes a la clase +1).")
        lines.append(f"Grado d={grado_criticoacc} apagado (ACC) - Nº Vectores soporte: {sum(clf_ACC_apagado_r1[grado_criticoacc].n_support_)} ({clf_ACC_apagado_r1[grado_criticoacc].n_support_[0]} pertenecientes a la clase -1, {clf_ACC_apagado_r1[grado_criticoacc].n_support_[1]} pertenecientes a la clase +1).")
    lines.append(f"Grado d=6 apagado (F1) - Nº Vectores soporte: {sum(clf_F1_apagado_r1[6].n_support_)} ({clf_F1_apagado_r1[6].n_support_[0]} pertenecientes a la clase -1, {clf_F1_apagado_r1[6].n_support_[1]} pertenecientes a la clase +1).")
    lines.append(f"Grado d=6 apagado (ACC) - Nº Vectores soporte: {sum(clf_ACC_apagado_r1[6].n_support_)} ({clf_ACC_apagado_r1[6].n_support_[0]} pertenecientes a la clase -1, {clf_ACC_apagado_r1[6].n_support_[1]} pertenecientes a la clase +1).")

fig, ax = plt.subplots(figsize=(12, len(lines) * 0.5 + 0.5))
ax.axis('off')
for i, line in enumerate(lines):
    ax.text(0, 1 - (i / len(lines)), line, fontsize=10, va='top', ha='left', transform=ax.transAxes)
plt.tight_layout()
plt.savefig("vectores_soporte_r1.png", dpi=150, bbox_inches='tight')
files.download("vectores_soporte_r1.png")
plt.show()